# 04. 상관관계와 산점도 분석

## 학습 목표
- Scatterplot으로 두 변수 간 관계 탐색
- Regplot, Lmplot으로 회귀선 추가
- Diamonds 데이터로 가격 결정 요인 분석
- 상관관계와 인과관계 구분

---

## 1. 왜 관계 분석이 중요한가?

**제조업 실무 사례:**
- 투입량-산출량 관계 → 최적 생산량 결정
- 온도-불량률 관계 → 공정 조건 최적화
- 무게-연비 관계 → 경량화 효과 정량화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# Diamonds 데이터 로드
diamonds = sns.load_dataset('diamonds')
print(f"전체 다이아몬드: {len(diamonds):,}개")
print(f"가격 범위: ${diamonds['price'].min():,} ~ ${diamonds['price'].max():,}")
diamonds.head()

## 2. 기본 산점도: 무게와 가격

**비즈니스 질문:** 캐럿(무게)이 가격에 미치는 영향은?

In [ ]:
# 샘플링 (가독성 위해)
diamonds_sample = diamonds.sample(1000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(diamonds_sample['carat'], diamonds_sample['price'],
           alpha=0.5, s=50, c='#3A86FF', edgecolors='black', linewidth=0.5)

ax.set_xlabel('캐럿 (무게)', fontsize=12)
ax.set_ylabel('가격 ($)', fontsize=12)
ax.set_title('다이아몬드 무게 vs 가격', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 상관계수 계산
corr = diamonds['carat'].corr(diamonds['price'])
print(f"상관계수: {corr:.3f}")

# 💡 인사이트: 강한 양의 상관관계(r=0.922) → 무게가 가격의 주요 결정 요인
# 💡 비선형 관계 확인 → 무게가 증가할수록 가격 상승폭이 더 커짐

## 3. Regplot: 회귀선 추가

**비즈니스 질문:** 무게를 기준으로 적정 가격을 예측할 수 있나?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Regplot 생성 (선형 회귀)
sns.regplot(data=diamonds_sample, x='carat', y='price', ax=ax,
            scatter_kws={'alpha': 0.4, 's': 40, 'color': '#FF006E'},
            line_kws={'color': '#FFBE0B', 'linewidth': 3})

ax.set_xlabel('캐럿 (무게)', fontsize=12)
ax.set_ylabel('가격 ($)', fontsize=12)
ax.set_title('다이아몬드 무게-가격 회귀 분석', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: 회귀선이 곡선 패턴을 충분히 반영 못함 → 다항 회귀 필요
# 💡 1캐럿 이상에서 잔차(residual) 커짐 → 다른 요인(품질) 영향 증가

## 4. 색상별 비교: Hue 활용

**비즈니스 질문:** 다이아몬드 등급(cut)에 따라 가격 패턴이 다른가?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Cut 등급별로 색상 구분
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
palette = sns.color_palette('viridis', n_colors=len(cut_order))

for i, cut in enumerate(cut_order):
    subset = diamonds_sample[diamonds_sample['cut'] == cut]
    ax.scatter(subset['carat'], subset['price'], 
              label=cut, alpha=0.6, s=50, c=[palette[i]],
              edgecolors='black', linewidth=0.5)

ax.set_xlabel('캐럿 (무게)', fontsize=12)
ax.set_ylabel('가격 ($)', fontsize=12)
ax.set_title('다이아몬드 컷 등급별 무게-가격 관계', fontsize=14, fontweight='bold')
ax.legend(title='컷 등급', fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: 동일 무게에서 Ideal 등급이 가장 높은 가격대 → 품질 프리미엄 존재
# 💡 하지만 무게 효과가 등급 효과보다 훨씬 강력함

## 5. Lmplot: FacetGrid 기반 회귀 플롯

**비즈니스 질문:** 투명도(clarity)별로 가격 결정 패턴이 다른가?

In [ ]:
# 주요 3개 등급만 선택
clarity_subset = diamonds_sample[diamonds_sample['clarity'].isin(['SI1', 'VS2', 'VVS2'])]

# Lmplot 생성
g = sns.lmplot(data=clarity_subset, x='carat', y='price', hue='clarity',
               height=5, aspect=1.5, scatter_kws={'alpha': 0.5, 's': 40},
               line_kws={'linewidth': 2.5}, palette='Set1')

g.set_axis_labels('캐럿 (무게)', '가격 ($)', fontsize=12)
g.fig.suptitle('투명도별 무게-가격 회귀선 비교', fontsize=14, fontweight='bold', y=1.02)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: VVS2(최고 투명도)의 회귀선 기울기가 가장 가파름 → 고품질일수록 무게 프리미엄 커짐
# 💡 SI1 등급은 같은 무게에서 가격 하락 → 품질 할인 효과

## 6. 3차원 관계: Size와 Color 동시 표현

**비즈니스 질문:** 깊이(depth)와 테이블(table)이 가격에 미치는 영향은?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Size=가격, Color=무게
scatter = ax.scatter(diamonds_sample['depth'], diamonds_sample['table'],
                    s=diamonds_sample['price']/10,  # 가격을 점 크기로
                    c=diamonds_sample['carat'],      # 무게를 색상으로
                    cmap='coolwarm', alpha=0.6, 
                    edgecolors='black', linewidth=0.5)

# Colorbar 추가
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('캐럿 (무게)', fontsize=11)

ax.set_xlabel('Depth (%)', fontsize=12)
ax.set_ylabel('Table (%)', fontsize=12)
ax.set_title('다이아몬드 치수 분석 (크기=가격, 색상=무게)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: Depth 60-63%, Table 54-58% 구간에 고가 다이아몬드 집중
# 💡 극단적 치수(depth<58% or >65%)는 무게가 커도 가격 낮음 → 최적 비율 존재

## 7. 실전 분석: 가격 결정 요인 우선순위

**비즈니스 질문:** 어떤 속성이 가격에 가장 큰 영향을 미치나?

In [ ]:
# 수치형 변수들의 상관계수 계산
numeric_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']
correlations = diamonds[numeric_cols + ['price']].corr()['price'].drop('price').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#06FFA5' if x > 0 else '#E63946' for x in correlations.values]
bars = ax.barh(correlations.index, correlations.values, 
               color=colors, edgecolor='black', linewidth=1.2)

# 수치 표시
for i, (attr, corr_val) in enumerate(zip(correlations.index, correlations.values)):
    ax.text(corr_val + 0.02, i, f'{corr_val:.3f}', 
            va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('상관계수', fontsize=12)
ax.set_ylabel('속성', fontsize=12)
ax.set_title('다이아몬드 속성별 가격 상관관계', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linewidth=1)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n변수별 상관계수:")
print(correlations)

# 💡 인사이트: 무게(carat) 0.922 >> 치수(x, y, z) 0.88 >> 비율(depth, table) -0.01
# 💡 Depth/Table은 가격과 거의 무관 → 최적 범위 내에서만 의미 있음
# 💡 크기 관련 변수들끼리 높은 상관관계 → 다중공선성 주의

## 8. 상관관계 vs 인과관계

**주의사항: 상관관계가 인과관계를 의미하지 않음**

In [ ]:
# 예시: 무게와 X 치수의 관계
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 좌측: Carat vs Price
sns.regplot(data=diamonds_sample, x='carat', y='price', ax=axes[0],
            scatter_kws={'alpha': 0.4, 's': 30}, color='#3A86FF')
axes[0].set_title(f'무게 vs 가격 (상관계수: {diamonds["carat"].corr(diamonds["price"]):.3f})',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('캐럿', fontsize=11)
axes[0].set_ylabel('가격 ($)', fontsize=11)

# 우측: X vs Price
sns.regplot(data=diamonds_sample, x='x', y='price', ax=axes[1],
            scatter_kws={'alpha': 0.4, 's': 30}, color='#FF006E')
axes[1].set_title(f'X 치수 vs 가격 (상관계수: {diamonds["x"].corr(diamonds["price"]):.3f})',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('X 치수 (mm)', fontsize=11)
axes[1].set_ylabel('가격 ($)', fontsize=11)

plt.tight_layout()
plt.show()

# 💡 인사이트: X 치수와 가격의 상관계수도 높지만(0.884), 실제로는 무게의 영향
# 💡 무게가 증가하면 치수도 증가 → Spurious Correlation (허위 상관)
# 💡 실무: 회귀 분석 시 다중공선성 검토 필수

---

## 📊 핵심 요약

### 코드 패턴 (30%)
```python
# 기본 산점도
ax.scatter(x, y, alpha=0.5, s=size, c=color)

# 회귀선 추가
sns.regplot(data=df, x='col1', y='col2', ax=ax)

# 그룹별 비교
sns.lmplot(data=df, x='col1', y='col2', hue='category')

# 3차원 표현
scatter = ax.scatter(x, y, s=size_var, c=color_var, cmap='viridis')
plt.colorbar(scatter)
```

### 도출된 인사이트 (70%)
1. **주요 요인**: 무게(r=0.922)가 가격의 85% 설명 → 캐럿당 가격 책정 전략 타당
2. **비선형 관계**: 무게 증가 시 가격 상승률도 증가 → 희소성 프리미엄
3. **품질 효과**: 동일 무게에서 Ideal 등급이 20-30% 고가 → 브랜딩 기회
4. **최적 비율**: Depth 60-63% 구간이 고가 형성 → 가공 표준 설정
5. **허위 상관**: X/Y/Z 치수는 무게의 대리변수 → 독립 변수 아님

### 제조업 적용
- **가격 책정**: 주요 속성 기반 가격 모델 구축
- **품질 관리**: 최적 스펙 범위 설정 (Depth 60-63%)
- **마케팅**: 무게 기준 세분화된 제품군 개발

### 주의사항
- 상관관계 ≠ 인과관계
- 다중공선성 검토 필수
- 이상치가 상관계수에 미치는 영향 크므로 전처리 중요